# GreenRoot Nursery Co. — Ad-hoc Analysis (Colab version)

**The question:** *"We think late shipments are dragging down reviews,
especially on the West Coast. Can you confirm and size it?"*

This is the Colab-adapted version of `02_adhoc_analysis.py`. Same setup
pattern as the reporting dashboard notebook: upload the zip, unzip it, and
use absolute paths instead of `../data`.

### Step 1 — Upload the project zip
Skip this if you already unzipped it earlier in this Colab session.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select plant-nursery-analytics.zip in the dialog

### Step 2 — Unzip it into Colab's workspace

In [ ]:
import zipfile, os

with zipfile.ZipFile("plant-nursery-analytics.zip", "r") as z:
    z.extractall(".")

os.listdir("plant-nursery-analytics")

### Step 3 — Imports and path setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import os

plt.style.use("seaborn-v0_8-whitegrid")

DATA = "plant-nursery-analytics/data"
CHARTS = "plant-nursery-analytics/charts"
os.makedirs(CHARTS, exist_ok=True)

orders = pd.read_csv(f"{DATA}/orders.csv", parse_dates=["date"])
reviewed = orders.dropna(subset=["review_score"]).copy()

orders.head()

### Q1 — Does on-time delivery actually correlate with review score?

We compare average review scores for on-time vs. late orders, and run an
independent-samples t-test (`scipy.stats.ttest_ind`) to check whether the
difference is statistically significant or could just be noise.

In [ ]:
by_otd = reviewed.groupby("on_time")["review_score"].agg(["mean", "count"])
t_stat, p_val = stats.ttest_ind(
    reviewed.loc[reviewed.on_time, "review_score"],
    reviewed.loc[~reviewed.on_time, "review_score"],
    equal_var=False,
)

fig, ax = plt.subplots(figsize=(6, 4.5))
bars = ax.bar(["On Time", "Late"], by_otd["mean"], color=["#2e7d32", "#c62828"])
for b, v in zip(bars, by_otd["mean"]):
    ax.text(b.get_x() + b.get_width()/2, v + 0.02, f"{v:.2f}", ha="center", fontsize=11)
ax.set_ylim(0, 5)
ax.set_ylabel("Avg. Review Score")
ax.set_title(f"Review Score: On-Time vs. Late Delivery\n(t={t_stat:.1f}, p={p_val:.1e})")
fig.tight_layout()
fig.savefig(f"{CHARTS}/05_review_vs_ontime.png", dpi=140)
plt.show()

### Q2 — Is the West Coast specifically worse, and why?

We split orders into "West Coast" (CA, WA, AZ) vs. "Rest of US" and compare
both on-time delivery rate and average review score side by side.

In [ ]:
reviewed["region"] = reviewed["customer_state"].map(
    lambda s: "West Coast" if s in ("CA", "WA", "AZ") else "Rest of US"
)
region_otd = orders.groupby(orders["customer_state"].map(
    lambda s: "West Coast" if s in ("CA", "WA", "AZ") else "Rest of US"
))["on_time"].mean() * 100
region_review = reviewed.groupby("region")["review_score"].mean()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))
ax1.bar(region_otd.index, region_otd.values, color=["#8d6e63", "#558b2f"])
ax1.set_title("On-Time Rate: West Coast vs. Rest of US")
ax1.set_ylabel("%")
ax1.set_ylim(80, 100)
for i, v in enumerate(region_otd.values):
    ax1.text(i, v + 0.3, f"{v:.1f}%", ha="center")

ax2.bar(region_review.index, region_review.values, color=["#8d6e63", "#558b2f"])
ax2.set_title("Avg Review Score: West Coast vs. Rest of US")
ax2.set_ylim(3.5, 5)
for i, v in enumerate(region_review.values):
    ax2.text(i, v + 0.02, f"{v:.2f}", ha="center")
fig.tight_layout()
fig.savefig(f"{CHARTS}/06_west_coast_deep_dive.png", dpi=140)
plt.show()

### Q3 — Root cause: is it specific supplier regions serving CA/WA/AZ?

We bring in the SKU catalog to see which `supplier_region` each product ships
from, then look only at West Coast orders to see whether on-time performance
varies by supplier region — this is what pinpoints the actual bottleneck.

In [ ]:
skus = pd.read_csv(f"{DATA}/skus.csv")
merged = orders.merge(skus[["sku", "supplier_region"]], on="sku")
merged["region"] = merged["customer_state"].map(
    lambda s: "West Coast" if s in ("CA", "WA", "AZ") else "Rest of US"
)
pivot = merged[merged.region == "West Coast"].groupby("supplier_region")["on_time"].mean().sort_values() * 100

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.barh(pivot.index, pivot.values, color="#558b2f")
ax.set_xlabel("On-Time Rate for West Coast Orders (%)")
ax.set_title("Root Cause: On-Time Rate by Supplier Region\n(West Coast customers only)")
for b, v in zip(bars, pivot.values):
    ax.text(v + 0.3, b.get_y() + b.get_height()/2, f"{v:.1f}%", va="center")
fig.tight_layout()
fig.savefig(f"{CHARTS}/07_root_cause_supplier_region.png", dpi=140)
plt.show()

### Sizing the business impact

Finally, we estimate how much revenue is actually exposed to this problem,
and compare 1–2 star review rates for late vs. on-time West Coast orders as
a rough proxy for the customer-experience cost.

In [ ]:
wc_orders = orders[orders.customer_state.isin(["CA", "WA", "AZ"])]
wc_late_share = 1 - wc_orders["on_time"].mean()
wc_revenue = (wc_orders["quantity"] * wc_orders["unit_price"]).sum()

# rough proxy: how much worse are low-star reviews for late vs on-time West Coast orders
low_review_rate_late = reviewed[~reviewed.on_time & reviewed.customer_state.isin(["CA","WA","AZ"])]["review_score"].lt(3).mean()
low_review_rate_ontime = reviewed[reviewed.on_time & reviewed.customer_state.isin(["CA","WA","AZ"])]["review_score"].lt(3).mean()

print("=== AD-HOC FINDINGS ===")
print(f"On-time delivery -> avg review: {by_otd.loc[True,'mean']:.2f} | Late -> {by_otd.loc[False,'mean']:.2f}  (p={p_val:.1e})")
print(f"West Coast on-time rate: {region_otd['West Coast']:.1f}% vs Rest of US: {region_otd['Rest of US']:.1f}%")
print(f"Root cause: Southeast-sourced SKUs shipped to West Coast are the primary driver")
print(f"West Coast 1-2 star review rate - late orders: {low_review_rate_late:.1%} vs on-time: {low_review_rate_ontime:.1%}")
print(f"West Coast revenue exposed to this gap (18mo): \${wc_revenue:,.0f}, {wc_late_share:.1%} shipped late")

### Step 4 (optional) — download the generated charts back to your computer

In [ ]:
import shutil
from google.colab import files as colab_files

shutil.make_archive("adhoc_charts_output", "zip", CHARTS)
colab_files.download("adhoc_charts_output.zip")